# 03.3 — GenRec Explanation Quality Evaluation

**RecSys 2026 Tutorial:** Choosing Between Explainable, Retrieval-Augmented, and LLM-Native Recommenders

## Why this notebook exists

Notebooks 01 and 02 evaluate explanation quality (relevance, specificity, consistency,
hallucination) for the Explainable and RAG paradigms. The generative paradigm produces
only a recommended title — no native explanation — so the §5.3 table in the report had
an empty Generative column.

This notebook closes that gap. For each test user we:
1. Load the title the fine-tuned QLoRA model already recommended (from NB 03.1's checkpoint).
2. Ask the **same base Qwen2.5-3B-Instruct** (the model NB01 and NB02 use for explanation)
   to justify that recommendation given the user's history. Using the base model — not
   the QLoRA adapter — keeps the explainer LLM identical across all three paradigms; only
   the upstream recommendation differs.
3. Run the same four explanation-quality metrics from `tutorial_utils.evaluate_explanations`.

**Requirements:** Colab A100 (recommended) or L4. Reads the inference checkpoint from
`03_1_genrec_qlora_finetuning.ipynb`.

**Output:** `generative_explanation_results.json` with the four metrics.

## 0. Setup

In [13]:
!pip install -q transformers accelerate sentence-transformers datasets torch

In [14]:
# ──────────────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────────────
SMOKE_TEST = False   # 3 users + 2 consistency users — flip to False for the full run

if SMOKE_TEST:
    N_EXPLAIN = 3
    N_CONSISTENCY = 2
    print('*** SMOKE TEST: 3 users + 2 consistency users ***')
else:
    N_EXPLAIN = 100      # matches NB 01 and 02
    N_CONSISTENCY = 20   # matches NB 01
    print(f'*** FULL RUN: {N_EXPLAIN} users for main metrics, {N_CONSISTENCY} for consistency ***')

*** FULL RUN: 100 users for main metrics, 20 for consistency ***


In [15]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [16]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = '/content/drive/MyDrive/Foundations of Large Language Models/Final Project'
DATA_DIR = f'{DRIVE_ROOT}/data'
RESULTS_DIR = f'{DRIVE_ROOT}/results'
GENREC_CHECKPOINT = f'{RESULTS_DIR}/genrec_finetuned_inference_checkpoint.pkl'

assert os.path.exists(GENREC_CHECKPOINT), \
    f'Missing {GENREC_CHECKPOINT}. Run notebook 03.1 first to populate fine-tuned predictions.'
print(f'Found inference checkpoint: {GENREC_CHECKPOINT}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found inference checkpoint: /content/drive/MyDrive/Foundations of Large Language Models/Final Project/results/genrec_finetuned_inference_checkpoint.pkl


## 1. Load data and cached fine-tuned recommendations

We pull each user's already-generated recommended title from NB 03.1's checkpoint, so we
don't waste compute re-generating titles.

In [17]:
import json
import pickle
import numpy as np

with open(f'{DATA_DIR}/shared_data.pkl', 'rb') as f:
    shared = pickle.load(f)
item_titles = shared['item_titles']
test_ground_truth = shared['test_ground_truth']
user_histories = shared['user_histories']

with open(GENREC_CHECKPOINT, 'rb') as f:
    ckpt = pickle.load(f)
generated_titles_dict = ckpt['generated_titles_dict']

print(f'Test users: {len(test_ground_truth)}')
print(f'Catalog items: {len(item_titles)}')
print(f'Fine-tuned predictions available: {len(generated_titles_dict)}')

# Build the eval user list: users with a non-empty generated title and known history
candidate_users = [
    uid for uid in generated_titles_dict
    if generated_titles_dict[uid]
    and generated_titles_dict[uid][0].strip()
    and uid in user_histories
    and len(user_histories[uid]) >= 1
]
eval_users = candidate_users[:N_EXPLAIN]
consistency_users = candidate_users[:N_CONSISTENCY]
print(f'Selected {len(eval_users)} users for main metrics, {len(consistency_users)} for consistency')

Test users: 7288
Catalog items: 6117
Fine-tuned predictions available: 7288
Selected 100 users for main metrics, 20 for consistency


In [18]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Use the BASE model (no QLoRA adapter) — same explainer LLM as NB01 and NB02.
# The QLoRA adapter was tuned for title-only output, so it's not the right
# instruction-follower for explanation generation.
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()

print(f'Loaded base {MODEL_ID}. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loaded base Qwen/Qwen2.5-3B-Instruct. VRAM: 6.27 GB


## 2. Explanation generation

Prompt template asks for a grounded 2-3 sentence justification that references specific
books from the user's history. We deliberately ask for *concrete* references to make
hallucination detectable downstream.

In [19]:
import time

RECENT_N = 10   # number of history titles to include in the prompt (most recent)


def build_prompt(history_titles, recommended_title):
    recent = history_titles[-RECENT_N:]
    history_block = '\n'.join(f'- {t}' for t in recent)
    user_msg = (
        f"A user has recently read these books (most recent last):\n"
        f"{history_block}\n\n"
        f'Based on this reading history, we are recommending: "{recommended_title}"\n\n'
        f'Write a brief 2-3 sentence explanation for why this user would enjoy this '
        f'recommendation. Reference at least one specific book from their history. '
        f'Be concrete — avoid generic phrases like "a great read" or "many people enjoy this".'
    )
    messages = [
        {'role': 'system', 'content': 'You are a recommender system explaining book recommendations.'},
        {'role': 'user', 'content': user_msg},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_explanation(prompt_text, do_sample=False, max_new_tokens=180):
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
    )
    if do_sample:
        gen_kwargs.update(do_sample=True, temperature=0.7, top_p=0.9)
    else:
        gen_kwargs.update(do_sample=False)
    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)
    text = tokenizer.decode(out[0, input_len:], skip_special_tokens=True).strip()
    return text

In [20]:
# Sanity check on the first eval user
uid = eval_users[0]
hist_idx = user_histories[uid][-RECENT_N:]
hist_titles = [item_titles.get(i, '?') for i in hist_idx]
rec_title = generated_titles_dict[uid][0]

prompt = build_prompt(hist_titles, rec_title)
print('--- PROMPT (first user) ---')
print(prompt[-500:])
print('\n--- GENERATED EXPLANATION ---')
t0 = time.time()
expl = generate_explanation(prompt)
print(expl)
print(f'\nlatency: {time.time() - t0:.1f}s')

--- PROMPT (first user) ---
commendations.<|im_end|>
<|im_start|>user
A user has recently read these books (most recent last):
- Die Trying (Jack Reacher Book 2)
- Gone Girl: A Novel

Based on this reading history, we are recommending: "The Nightingale: A Novel"

Write a brief 2-3 sentence explanation for why this user would enjoy this recommendation. Reference at least one specific book from their history. Be concrete — avoid generic phrases like "a great read" or "many people enjoy this".<|im_end|>
<|im_start|>assistant


--- GENERATED EXPLANATION ---
Given the user's recent enjoyment of "Die Trying" by Lee Child, which features intense action and psychological depth, "The Nightingale" offers a compelling continuation of such themes with its dual narrative structure that delves into the lives of two sisters during World War II France. The intricate plot and character development in "The Nightingale" align well with the user's taste for complex storytelling and suspenseful narratives.

## 3. Main metrics — one explanation per user (deterministic)

Resumable from a checkpoint.

In [21]:
MAIN_CKPT = f'{RESULTS_DIR}/genrec_explanation_main_checkpoint.pkl'

if os.path.exists(MAIN_CKPT) and not SMOKE_TEST:
    with open(MAIN_CKPT, 'rb') as f:
        state = pickle.load(f)
    explanations = state['explanations']
    latencies_main = state['latencies']
    print(f'Resumed: {len(explanations)} users already done.')
else:
    explanations = {}     # {uid: {'item_idx': int, 'text': str}}
    latencies_main = []

remaining = [u for u in eval_users if u not in explanations]
print(f'Total: {len(eval_users)}, Remaining: {len(remaining)}')

for i, uid in enumerate(remaining):
    hist_idx = user_histories[uid][-RECENT_N:]
    hist_titles = [item_titles.get(idx, '') for idx in hist_idx if idx in item_titles]
    rec_title = generated_titles_dict[uid][0]
    rec_item_idx = -1  # generated title may not match catalog; record raw text only

    t0 = time.time()
    text = generate_explanation(build_prompt(hist_titles, rec_title), do_sample=False)
    latencies_main.append(time.time() - t0)

    explanations[uid] = {'item_idx': rec_item_idx, 'text': text, 'recommended_title': rec_title}

    if (i + 1) % 25 == 0 or (i + 1) == len(remaining):
        if not SMOKE_TEST:
            with open(MAIN_CKPT, 'wb') as f:
                pickle.dump({'explanations': explanations, 'latencies': latencies_main}, f)
        print(f'  {len(explanations)}/{len(eval_users)} | '
              f'mean latency: {np.mean(latencies_main[-25:]):.1f}s')

print(f'\nDone. {len(explanations)} explanations generated.')
if latencies_main:
    print(f'Mean latency: {np.mean(latencies_main):.1f}s/explanation')

Total: 100, Remaining: 100
  25/100 | mean latency: 4.0s
  50/100 | mean latency: 4.1s
  75/100 | mean latency: 4.4s
  100/100 | mean latency: 4.4s

Done. 100 explanations generated.
Mean latency: 4.2s/explanation


## 4. Consistency — 3 explanations per user with sampling

In [22]:
CONS_CKPT = f'{RESULTS_DIR}/genrec_explanation_consistency_checkpoint.pkl'

if os.path.exists(CONS_CKPT) and not SMOKE_TEST:
    with open(CONS_CKPT, 'rb') as f:
        state = pickle.load(f)
    consistency_texts = state['consistency_texts']
    latencies_cons = state['latencies']
    print(f'Resumed: {len(consistency_texts)} users already done.')
else:
    consistency_texts = {}
    latencies_cons = []

remaining = [u for u in consistency_users if u not in consistency_texts]
print(f'Total: {len(consistency_users)}, Remaining: {len(remaining)}')

for i, uid in enumerate(remaining):
    hist_idx = user_histories[uid][-RECENT_N:]
    hist_titles = [item_titles.get(idx, '') for idx in hist_idx if idx in item_titles]
    rec_title = generated_titles_dict[uid][0]

    runs = []
    prompt = build_prompt(hist_titles, rec_title)
    for run_i in range(3):
        t0 = time.time()
        text = generate_explanation(prompt, do_sample=True)
        latencies_cons.append(time.time() - t0)
        runs.append(text)
    consistency_texts[uid] = runs

    if (i + 1) % 5 == 0 or (i + 1) == len(remaining):
        if not SMOKE_TEST:
            with open(CONS_CKPT, 'wb') as f:
                pickle.dump({'consistency_texts': consistency_texts, 'latencies': latencies_cons}, f)
        print(f'  {len(consistency_texts)}/{len(consistency_users)} | '
              f'recent mean latency: {np.mean(latencies_cons[-15:]):.1f}s')

print(f'\nDone. {len(consistency_texts)} users × 3 runs each.')
if latencies_cons:
    print(f'Mean latency: {np.mean(latencies_cons):.1f}s/generation')

Total: 20, Remaining: 20
  5/20 | recent mean latency: 4.5s
  10/20 | recent mean latency: 4.3s
  15/20 | recent mean latency: 4.8s
  20/20 | recent mean latency: 4.1s

Done. 20 users × 3 runs each.
Mean latency: 4.4s/generation


## 5. Evaluate using the shared metric definitions

We call the exact same `evaluate_explanations` function NB01 and NB02 use, so the metric
definitions are identical across all three paradigms.

In [23]:
# Inline the metric implementations (a copy of tutorial_utils.evaluate_explanations so
# this notebook is self-contained on Colab without uploading tutorial_utils.py)
import re, math
from sentence_transformers import SentenceTransformer

MIN_TITLE_LENGTH = 4


def _cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))


def explanation_relevance(explanation, history_titles):
    if not explanation or not history_titles:
        return 0.0
    e = explanation.lower()
    return sum(1 for t in history_titles if t.lower() in e) / len(history_titles)


def explanation_specificity(explanation, item_title):
    if not explanation:
        return 0.0
    words = explanation.split()
    if len(words) < 10:
        return 0.1
    item_mentioned = 1.0 if item_title and item_title.lower() in explanation.lower() else 0.0
    length_score = min(len(words) / 50.0, 1.0)
    return 0.5 * item_mentioned + 0.5 * length_score


def explanation_hallucination(explanation, item_title, history_titles, all_titles):
    if not explanation:
        return 0.0
    e = explanation.lower()
    valid = set(t.lower() for t in history_titles) | ({item_title.lower()} if item_title else set())
    mentioned = 0
    halluc = 0
    for t in all_titles:
        tl = t.lower()
        if len(tl) < MIN_TITLE_LENGTH:
            continue
        if re.search(r'\b' + re.escape(tl) + r'\b', e):
            mentioned += 1
            if tl not in valid:
                halluc += 1
    return halluc / mentioned if mentioned > 0 else 0.0


def explanation_consistency_lexical(texts):
    if len(texts) < 2:
        return 1.0
    scores = []
    for i in range(len(texts)):
        for j in range(i + 1, len(texts)):
            a = set(texts[i].lower().split())
            b = set(texts[j].lower().split())
            if a and b:
                scores.append(len(a & b) / len(a | b))
    return float(np.mean(scores)) if scores else 0.0


def explanation_consistency_embedding(texts, encoder):
    if len(texts) < 2:
        return 1.0
    embs = encoder.encode(texts)
    scores = []
    for i in range(len(embs)):
        for j in range(i + 1, len(embs)):
            scores.append(_cosine_sim(embs[i], embs[j]))
    return float(np.mean(scores)) if scores else 0.0


encoder = SentenceTransformer('all-MiniLM-L6-v2')
print(f'Loaded encoder: all-MiniLM-L6-v2')

# Build the catalog title set for the hallucination check
all_titles = set(item_titles.values())

# ── Compute the 4 metrics ──
rel_lex, rel_emb, spec, halluc = [], [], [], []
for uid, e in explanations.items():
    hist_titles = [item_titles.get(i, '') for i in user_histories.get(uid, []) if i in item_titles]
    item_title = e.get('recommended_title', '')
    text = e['text']

    rel_lex.append(explanation_relevance(text, hist_titles))
    spec.append(explanation_specificity(text, item_title))
    halluc.append(explanation_hallucination(text, item_title, hist_titles, all_titles))

    if hist_titles:
        history_text = '; '.join(hist_titles)
        embs = encoder.encode([text, history_text])
        rel_emb.append(_cosine_sim(embs[0], embs[1]))

# Consistency
cons_lex_scores = [explanation_consistency_lexical(t) for t in consistency_texts.values() if len(t) >= 2]
cons_emb_scores = [explanation_consistency_embedding(t, encoder) for t in consistency_texts.values() if len(t) >= 2]

results = {
    'relevance_lexical': float(np.mean(rel_lex)) if rel_lex else 0.0,
    'relevance_embedding': float(np.mean(rel_emb)) if rel_emb else 0.0,
    'specificity': float(np.mean(spec)) if spec else 0.0,
    'consistency_lexical': float(np.mean(cons_lex_scores)) if cons_lex_scores else 0.0,
    'consistency_embedding': float(np.mean(cons_emb_scores)) if cons_emb_scores else 0.0,
    'hallucination_rate': float(np.mean(halluc)) if halluc else 0.0,
}

print('\nGenerative paradigm — explanation quality:')
for k, v in results.items():
    print(f'  {k}: {v:.4f}')

print('\n--- Cross-paradigm comparison ---')
print(f'{"Metric":<25} {"Explainable":>12} {"RAG":>10} {"Generative":>12}')
print('-' * 65)
print(f'{"Relevance (lexical)":<25} {0.264:>12.3f} {0.222:>10.3f} {results["relevance_lexical"]:>12.3f}')
print(f'{"Relevance (embedding)":<25} {0.554:>12.3f} {0.602:>10.3f} {results["relevance_embedding"]:>12.3f}')
print(f'{"Specificity":<25} {0.710:>12.3f} {0.631:>10.3f} {results["specificity"]:>12.3f}')
print(f'{"Consistency (lexical)":<25} {0.358:>12.3f} {1.000:>10.3f} {results["consistency_lexical"]:>12.3f}')
print(f'{"Consistency (embedding)":<25} {0.861:>12.3f} {1.000:>10.3f} {results["consistency_embedding"]:>12.3f}')
print(f'{"Hallucination rate":<25} {0.321:>12.3f} {0.369:>10.3f} {results["hallucination_rate"]:>12.3f}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded encoder: all-MiniLM-L6-v2

Generative paradigm — explanation quality:
  relevance_lexical: 0.1985
  relevance_embedding: 0.5276
  specificity: 0.6798
  consistency_lexical: 0.2815
  consistency_embedding: 0.8255
  hallucination_rate: 0.3382

--- Cross-paradigm comparison ---
Metric                     Explainable        RAG   Generative
-----------------------------------------------------------------
Relevance (lexical)              0.264      0.222        0.199
Relevance (embedding)            0.554      0.602        0.528
Specificity                      0.710      0.631        0.680
Consistency (lexical)            0.358      1.000        0.282
Consistency (embedding)          0.861      1.000        0.825
Hallucination rate               0.321      0.369        0.338


## 6. Save

In [24]:
if SMOKE_TEST:
    print('*** SMOKE TEST: skipping save ***')
else:
    payload = {
        'paradigm': 'generative_explanation',
        'explainer_model': MODEL_ID + ' (base, no QLoRA adapter)',
        'recommender_model': 'Qwen2.5-3B QLoRA fine-tuned (from NB 03.1)',
        'n_main': len(explanations),
        'n_consistency_users': len(consistency_texts),
        'metrics': results,
        'latency_main_seconds_mean': float(np.mean(latencies_main)) if latencies_main else None,
        'latency_consistency_seconds_mean': float(np.mean(latencies_cons)) if latencies_cons else None,
    }

    with open(f'{RESULTS_DIR}/generative_explanation_results.json', 'w') as f:
        json.dump(payload, f, indent=2, default=str)

    # Also dump the raw explanation texts so we can do qualitative spot-checks later
    with open(f'{RESULTS_DIR}/generative_explanation_texts.pkl', 'wb') as f:
        pickle.dump({
            'explanations': explanations,
            'consistency_texts': consistency_texts,
        }, f)

    print(f'Saved to {RESULTS_DIR}/generative_explanation_results.json')

Saved to /content/drive/MyDrive/Foundations of Large Language Models/Final Project/results/generative_explanation_results.json
